In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages   
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated

In [3]:
class ChatState(BaseModel):
    messages:Annotated[list,add_messages]  ## add_messages will append the every new mssg in the list

In [4]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
## Node
def chatBotNode(state:ChatState) -> ChatState:
    res = llm.invoke(state.messages)
    state.messages = [res]  ## res will atomatically append in the state.messages list by add_messages
                            ## if we directly use .append(res) then I am manually modifying the state object(it is only in memory, unknown from langGraph working), But this is not the normal LangGraph pattern
                            ## when this node returns state update then add_message will automatically append these new update in Global state
    return state   ## every node returns a state update then LangGraph merges that update into the existing Global state

In [6]:
memory = InMemorySaver()

In [7]:
graph = StateGraph(ChatState)
graph.add_node("chatBot",chatBotNode)


## build Edge : we hv build the flow
graph.add_edge(START,"chatBot")
graph.add_edge("chatBot",END)



finalGraph = graph.compile(checkpointer=memory)

In [10]:
res = finalGraph.invoke(
    {"messages" : [{"role":"user", "content":"what ques i have asked"}]},
    {"configurable":{"thread_id":"1"}}
    )


res

{'messages': [HumanMessage(content='Hello, my name is nakul', additional_kwargs={}, response_metadata={}, id='5cd12bac-89d3-4e4c-8134-e8121ead8cf3'),
  AIMessage(content='Hello Nakul! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says "Hello, my name is nakul". Should greet them. Possibly ask how to help.'}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 78, 'total_tokens': 122, 'completion_time': 0.0623261, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.003655616, 'prompt_tokens_details': None, 'queue_time': 0.308841089, 'total_time': 0.065981716}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_24bfb4a850', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f250-cdaa-7230-b0c7-0124cc33c1c3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 44, 'total_tokens': 122, 'output_token_de

In [11]:
print(res["messages"][-1].content)

You’ve asked two things so far:

1. **“Hello, my name is nakul”** – a brief introduction.  
2. **“what ques i have asked”** – a request to recap the questions you’ve posed.  

Let me know if there’s anything else you’d like to add or discuss!
